# 11 — Agentic & Self-Correcting RAG with LangGraph (CRAG)

**By the end of this notebook, you can build a stateful LangGraph agent that grades retrieved docs, rewrites weak queries automatically, and stops safely at a retry cap.**

### Learning Objectives
- Understand Corrective RAG (CRAG) as a self-checking state graph rather than a blind straight pipe.
- Implement LangGraph `StateGraph` nodes (`retrieve`, `grade_documents`, `rewrite_query`, `generate`) with conditional edges.
- Enforce a hard `max_retries` loop guard so self-correction never runs into an infinite loop.

### Prerequisites
- Notebooks 00–10

## 1. Setup
Let's connect this notebook to our shared project helpers (`utils/models.py`), load `.env` (defaulting to free offline `local` mode), and initialize our LLM and embeddings.

In [1]:
# !pip install -q -r ../requirements.txt  # Uncomment if running in a fresh remote environment
import sys
from pathlib import Path
root = Path.cwd() if (Path.cwd() / 'utils').exists() else Path.cwd().parent
if str(root) not in sys.path: sys.path.insert(0, str(root))
from utils.models import get_llm, get_embeddings, resolve_project_root
project_root = resolve_project_root()
data_dir = project_root / 'data'
llm = get_llm()
embeddings = get_embeddings()
print(f'Ready! Root={project_root.name} | LLM={type(llm).__name__}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Ready! Root=rag-from-scratch-course | LLM=LocalGroundedChatModel


## 2. Concept: From Straight Pipe to Self-Correcting State Machine

A basic RAG pipe blindly passes *whatever* the retriever finds straight into the LLM — even if the retrieved chunks are completely off-topic!

**Corrective RAG (CRAG)** with **LangGraph** acts like a conscientious research assistant:
1. **Retrieve** initial candidate chunks.
2. **Grade** each chunk cleanly (`yes` = relevant, `no` = off-topic noise) and throw away off-topic chunks.
3. **Decide (Conditional Edge)**: Did at least 1 relevant chunk survive?
   - **Yes** -> go straight to **Generate**!
   - **No** (and `retries < 2`) -> **Rewrite Query** cleanly and loop back to Retrieve!
   - **No** (and `retries >= 2`) -> Exit cleanly to Generate (which replies *"I don't know"* honestly!).

```
[START] --> [retrieve] --> [grade_documents] --(relevant docs found?)--+--> YES --> [generate] --> [END]
                                      ^                                  |
                                      |                                  +-- NO (retries < 2) --> [rewrite_query]
                                      +-------------------------------------------------------------+
```

## 3. Step-by-Step Build

### Step 1: Index Corpus & Define LangGraph `GraphState`
Let's create our vector store and define a TypedDict holding every variable flowing between graph nodes (`question`, `docs`, `retries`, `answer`).

In [2]:
from typing import List, TypedDict
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from utils.helpers import load_corpus_documents

chunks = RecursiveCharacterTextSplitter(chunk_size=340, chunk_overlap=40).split_documents(load_corpus_documents(data_dir))
retriever = InMemoryVectorStore.from_documents(chunks, embeddings).as_retriever(search_kwargs={"k": 3})

class GraphState(TypedDict):
    question: str
    docs: List[Document]
    retries: int
    answer: str
        

> **What just happened?**
> `GraphState` serves as the shared whiteboard updated cleanly by each LangGraph node.

### Step 2: Define `retrieve_node` and Structured `grade_node`
First we define retrieval, then a Pydantic structured grader that filters out any irrelevant documents before generation.

In [3]:
from pydantic import BaseModel, Field

class GradeDoc(BaseModel):
    relevant: bool = Field(description="True if document helps answer question")

grader = llm.with_structured_output(GradeDoc)
retrieve_node = lambda state: {"docs": retriever.invoke(state["question"])}

def grade_node(state: GraphState):
    kept = [d for d in state["docs"] if grader.invoke(f"QUESTION: {state['question']}\nDOCUMENT: {d.page_content}").relevant]
    return {"docs": kept}
        

> **What just happened?**
> Off-topic chunks get filtered right here instead of polluting the prompt.

### Step 3: Define `rewrite_node`, `generate_node`, and Retry Guard
When zero chunks pass grading, `rewrite_node` improves wording and increments `retries`; `decide_next` caps retries at `<= 2`.

In [4]:
def rewrite_node(state: GraphState):
    better_q = llm.invoke(f"Rewrite the query as a crisp standalone question:\n{state['question']}").content
    return {"question": better_q, "retries": state["retries"] + 1}

def generate_node(state: GraphState):
    ctx = "\n---\n".join(d.page_content for d in state["docs"])
    ans = llm.invoke(f"<context>\n{ctx}\n</context>\nQuestion: {state['question']}").content
    return {"answer": ans}

def decide_next(state: GraphState):
    if state["docs"] or state["retries"] >= 2:
        return "generate"
    return "rewrite"
        

> **What just happened?**
> Checking `state['retries'] >= 2` guarantees every question terminates safely in finite steps.

### Step 4: Compile & Execute the Corrective RAG StateGraph
Let's wire nodes and conditional edges using LangGraph `StateGraph` and trace both an answerable query and an unanswerable gap query.

In [5]:
from langgraph.graph import END, START, StateGraph
g = StateGraph(GraphState)
for name, fn in [("retrieve", retrieve_node), ("grade", grade_node), ("rewrite", rewrite_node), ("generate", generate_node)]:
    g.add_node(name, fn)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "grade")
g.add_conditional_edges("grade", decide_next, {"rewrite": "rewrite", "generate": "generate"})
g.add_edge("rewrite", "retrieve")
g.add_edge("generate", END)
crag_app = g.compile()

res_ok = crag_app.invoke({"question": "What is ZeroGravity failover timeout?", "docs": [], "retries": 0, "answer": ""})
res_miss = crag_app.invoke({"question": "What is Novasphere's NASDAQ stock ticker?", "docs": [], "retries": 0, "answer": ""})
print(f"OK query  -> retries={res_ok['retries']} | ans={res_ok['answer'][:95]}")
print(f"Gap query -> retries={res_miss['retries']} | ans={res_miss['answer'][:95]}")
        

OK query  -> retries=0 | ans=Based on the provided context: ### How does ZeroGravity Failover work, and which plan includes 
Gap query -> retries=2 | ans=I don't know based on the provided context. No usable documents were supplied.


> **What just happened?**
> On the valid query it succeeded immediately (`retries=0`); on the missing-data query it retried cleanly up to cap (`retries=2`) and returned a grounded refusal!

## 4. Try It Yourself
Experiment by editing the starter ideas below before peeking at the solution!

1. **Test error troubleshooting via CRAG** — *Hint:* Invoke `crag_app` on `ERR_QUANTUM_TIMEOUT` steps.
2. **Verify termination cap** — *Hint:* Trace `res_miss['retries']` is strictly `<= 2` (max 3 total search attempts).

<details>
<summary><strong>Click to reveal sample solution</strong></summary>

Run the cell below to verify your solution.
</details>

In [6]:
err_out = crag_app.invoke({"question": "How do I resolve ERR_QUANTUM_TIMEOUT?", "docs": [], "retries": 0, "answer": ""})
assert err_out["retries"] <= 2
print("Self-corrected error guide:", err_out["answer"])
        

Self-corrected error guide: Based on the provided context: ### How do I resolve error code `ERR_QUANTUM_TIMEOUT`? `ERR_QUANTUM_TIMEOUT` occurs when the edge agent cannot complete its high-frequency QuantumCache v4 state handshake with the local coordinator. Follow these three steps in order: 1. Verify that host and perimeter firewalls permit inbound and outbound UDP traffic on **port 9872**.


## 5. Common Mistakes
- Omitting a retry cap (`retries >= max`) in conditional edges — creates infinite API loops and runaway costs.
- Rewriting queries blindly without passing why retrieval failed.
- Dropping retrieved docs if grader is overly strict without falling back gently.

## 6. Recap
- LangGraph models agentic loops (`retrieve -> grade -> rewrite`) cleanly using explicit state.
- Grading retrieved chunks before answering prevents irrelevant context from reaching the LLM.
- Every cyclic agent graph MUST have a hard termination guard (`retries >= 2`).

## 7. Next Notebook
Next up (`12_evaluating_rag.ipynb`): Stop guessing if tweaks helped — measure Hit Rate@k, MRR, Faithfulness, and run controlled Ablation Studies!

## 8. Video Chapter Outline
1. **Limitations of Linear RAG Chains** — Why ungraded retrieval blindly trusts bad search results.
2. **Corrective RAG Architecture** — Grade -> Decide -> Rewrite -> Retry state diagram.
3. **Building LangGraph State & Nodes** — Structured grading + loop cap implementation.
4. **Tracing Valid vs Missing Queries** — Seeing real retry counters & grounded refusals live.
5. **Safety Guards & Recap** — Preventing runaway graph cycles.